# L6

Na tych ?wiczeniach om?wimy:
* model evaluation,
* model selection,
* data augmentation.


## Cross validation - przypomnienie

Najprostszym sposobem na ocen? skuteczno?ci modelu jest podzielenie danych w losowy spos?b na zbiory trenuj?cy i testowy, wytrenowanie modelu na zbiorze trenuj?cym i przetestowanie na zbiorze testowym. Cross validation stosujemy w celu zmniejszenia wariancji estymatora losowego podzia?u.

### Zasada dzia?ania

Robimy jeden du?y podzia? - dzielimy ca?y zbi?r na $k$ roz??cznych podzbior?w i potem $k$ razy zbi?r trenuj?cy sk?adamy z $k-1$ kawa?k?w, a ostatniego u?ywamy jako zbi?r testowy. Dzi?ki temu ka?dy element $D$ b?dzie dok?adnie raz w zbiorze testowym.

<img src="figures/L3/K-fold_cross_validation_EN.jpg">

### Leave one out

Szczeg?lnym przypadkiem jest podej?cie leave one out, gdzie zbi?r testowy jest zawsze jednoelementowy ($k$ jest r?wne liczbie przyk?ad?w).

## Model evaluation

Model evaluation to ca?a rodzina algorytm?w estymowania skuteczno?ci nauczonego modelu. Oczywi?cie interesuje nas to, jak model poradzi sobie na danych, kt?re b?d? pojawia?y si? "w prawdziwym ?wiecie", a wi?c na danych pochodz?cych z tego samego ?r?d?a, co zbi?r treningowy, lecz niekoniecznie identycznych (dane takie same, a nie te same).

### Dane i.i.d.

W najprostszym przypadku zak?adamy, ?e dane pochodz? z tego samego rozk?adu, wi?c mo?emy np.:
* podzieli? dane losowo na zbi?r trenuj?cy i testowy,
* wykona? powy?sze wielokrotnie i u?redni? wynik,
* u?y? cross validation,
* u?y? leave one out.

Za ka?dym razem model jest uczony od nowa na zbiorze treningowym, a testowany tylko na danych ze zbioru testowego.

W wypadku problem?w klasyfikacji podzia? train/test mo?emy wykona? w spos?b "stratified" - je?li np. zbi?r testowy stanowi 30% wszystkich danych, to dodatkowo wymuszamy, aby 30% danych z **ka?dej klasy** znalaz?o si? w zbiorze testowym. Jest to szczeg?lnie wa?ne w wypadku, gdy niekt?re klasy s? ma?o liczebne w naszych danych - w ten spos?b zapobiegamy sytuacji, w kt?rej ca?a klasa zostanie pomini?ta w zbiorze treningowym lub testowym.

### Dane nie i.i.d.

Oczywi?cie w praktyce nie zawsze mo?na sensownie uargumentowa? i.i.d. Przyk?adem niech b?dzie zbi?r MNIST - ka?dy cz?owiek ma nieco inny charakter pisma, a zadanie polega na nauczeniu modelu, kt?ry dobrze radzi sobie na danych pochodz?cych od nowych ludzi. Dlatego te? tw?rca MNISTa dostarcza podzia? train/test, kt?ry wykonany jest wzgl?dem ludzi (w zbiorze treningowym znajduj? si? cyfry napisane przez ok. 250 os?b, ??cznie 60 tys. przyk?ad?w, w zbiorze testowym znajduje si? 10 tys. przyk?ad?w pochodz?cych od innych os?b). Dzi?ki temu sprawdzamy, czy model faktycznie generalizuje na nowy charakter pisma, a nie tylko zapami?tuje poszczeg?lne osoby i ich styl.

W tym notebooku b?dziemy dla prostoty u?ywali wszystkich danych ??cznie, ale nale?y pami?ta?, co w ten spos?b tak naprawd? sprawdzamy.

### Szeregi czasowe

Bardzo cz?sto dane pochodz? z szeregu czasowego, tzn. wiersze s? postaci ($\mathbf{x}$, $y$, timestamp). Je?li zadanie polega na generalizacji w czasie - czyli oczekujemy, ?e nasz model b?dzie dzia?a? dobrze w przysz?o?ci - to przy ewaluacji musimy wykona? podzia? train/test ze wzgl?du na czas (train to wszystkie dane przed pewn? ustalon? dat?, test to dane p??niejsze). Temat szereg?w czasowych wykracza poza ramy tego notebooka. 

## Model selection

Modele opisane s? dwoma zestawami liczb: parametrami i hiperparametrami.

### Parametry

Parametry to zmienne, kt?re wyznaczane s? algorytmicznie podczas uczenia modelu (jawnym wzorem, metodami iteracyjnymi itp.). Przyk?adem s? wagi w regresji liniowej, wagi w sieciach neuronowych, support vectors w SVMach, progi w drzewach decyzyjnych.

### Hiperparametry

W odr??nieniu od parametr?w, hiperparametry musz? by? dobrane r?cznie przed rozpocz?ciem uczenia. Hiperparametry kontroluj? dzia?anie modelu, pozwalaj? manipulowa? overfittingiem itp. Przyk?ady: sta?a C w regresji liniowej, architektura sieci neuronowej (liczba i rozmiar warstw, funkcje aktywacji), wymiar podprzestrzeni w PCA, liczba drzew w Random Forest.

### Motywacja

W za?o?eniu hiperparametry pozwalaj? w??czy? cz?owieka w proces uczenia modelu - cz?owiek potrafi dobra? hiperparametry, poniewa? posiada intuicj? dotycz?c? prawdziwego ?wiata i, w przeciwie?stwie do maszyny, rozumie dany task znacznie g??biej oraz potrafi poprawnie zinterpretowa? dzia?anie hiperparametr?w. Natomiast znalezieniem parametr?w, co jest zadaniem ?mudnym, ale dobrze zdefiniowanym, mo?e zaj?? si? komputer.

Oczywi?cie to wszystko nieprawda - cz?owiek wcale nie ma lepszej intuicji, a zezwolenie na r?czny dob?r hiperparametr?w prowadzi do r??nych patologii (np. testujemy dziesi?tki tysi?cy zestaw?w hiperparamet?w i publikujemy najlepszy wynik - zamiast dosta? lepszy model zoverfitowali?my si? do danych).

Dlatego przyjmijmy na t? chwil? inn? definicj? - hiperparametry (tym razem z definicji!) r??ni? si? od parametr?w tym, ?e nie da si? ich zoptymalizowa? gradientowo lub jawnym wzorem. Model selection to po prostu algorytm doboru hiperparametr?w.

### Przyk?ad

Za???my, ?e mamy zbi?r (X_train, y_train) oraz klas? Model, kt?ra przyjmuje w konstruktorze zestaw hiperparametr?w. Mo?emy zrobi? nast?puj?co:
* stworzy? list? zestaw?w hiperparametr?w i dla ka?dego stworzy? obiekt model = Model(zestaw_hiperparametr?w),
* oceni? wszystkie modele na danych (X_train, y_train),
* wybra? ten zestaw hiperparametr?w, dla kt?rego model uzyska? najlepszy wynik.

### Jak stworzy? list? zestaw?w hiperparametr?w

Zamienili?my problem r?cznego doboru hiperparametr?w na r?czne tworzenie listy zestaw?w hiperparametr?w - czy w takim razie wci?? nie oszukujemy? Troch? tak, ale na szcz??cie troch? mniej.

Jak w praktyce tworzy si? list? zestaw?w hiperparametr?w:
1. Grid search - dla ka?dego hiperparametru arbitralnie tworzymy list? jego mo?liwych warto?ci, a nast?pnie sprawdzamy wszystkie mo?liwe zestawy (problem: dla 10 hiperparametr?w i 10 mo?liwo?ci ka?dego z nich musimy wytrenowa? $10^{10}$ model?w).
2. Random grid search - robimy jak powy?ej, ale losujemy $n$ zestaw?w ($n$ dobieramy r?cznie, np. 43) zamiast sprawdza? wszystkie.
3. Random search - dla danego hiperparametru nie definiujemy listy jego mo?liwych warto?ci, ale rozk?ad, z kt?rego b?dziemy go losowali (np. losujemy $\alpha$ z rozk?adu jednostajnego na odcinku $[-5,5]$, a potem definiujemy hiperparametr regresji liniowej $C=10^{\alpha}$); nast?pnie zestaw hiperparametr?w tworzymy losuj?c ka?dy niezale?nie z podanego rozk?adu.

Podej?cie trzecie jest zdecydowanie najlepsze, co obrazuje poni?sza ilustracja [prosz? spr?bowa? wyt?umaczy?].

<img width=600 src="figures/L6/scikitlearn8.jpeg">

### * Optymalizacja bayesowska

Mo?na robi? jeszcze lepiej, ni? random search. Rozk?ady podane w random search traktujemy jako nasz? wiedz? a priori o po?o?eniu najlepszego zestawu hiperparametr?w, a nast?pnie po ka?dym losowaniu ewaluujemy model i uzyskany wynik w??czamy do naszej wiedzy. Szczeg??y s? nieco skomplikowane (musimy za?o?y? troch? dodatkowych w?asno?ci o tym, jak wynik modelu zale?y od po?o?enia w przestrzeni hiperparametr?w), natomiast zainteresowane osoby mog? spr?bowa? zapozna? si? np. z nast?puj?cymi implementacjami:
* http://hyperopt.github.io/hyperopt/
* https://github.com/msmbuilder/osprey

## Podsumowanie

Zak?adamy, ?e Model to klasa, a model to instancja. Model przyjmuje w konstruktorze wszystkie hiperparametry. Dane to (X,y). Model selection i model evaluation wykonujemy przy pomocy 3-fold cross validation. Ustalamy te? list? zestaw?w hiperparametr?w.

### Wa?na i subtelna uwaga

Techniki model evaluation mo?emy stosowa? tylko do konkretnych algorytm?w. Dlatego nie mo?emy zewaluowa? regresji liniowiej, tylko np. regresj? liniow? ze sta?? C=3. Nie mo?emy ewaluowa? klasy Model, tylko obiekt model.

Ale gdy zdecydujemy si? na pewien algorytm doboru hiperparametr?w, to mo?emy zapisa?:

(Model + model_selection) = metamodel

gdzie metamodel da si? fitowa? (parametry **oraz hiperparametry** zostan? dobrane automatycznie tylko na podstawie danych). W tym sensie mo?emy ewaluowa? klas? Model.

Warto te? zwr?ci? uwag?, ?e dla r??nych model_selection dostaniemy r??ne metamodele. Np. je?li raz za model_selection u?yjemy strategii cross validation z 3 foldami, a za drugim razem np. leave one out, to te dwa metamodele b?d? mia?y r??n? zasad? dzia?ania i przez to inny score (nawet je?li w obu przypadkach Model to np. regresja liniowa).

### Algorytmy

Aby zewaluowa? Model (metamodel):
1. Dzielimy trzy razy (X,y) na (X_train,y_train), (X_test,y_test).
2. Dla ka?dego takiego podzia?u:
    1. Dzielimy trzy razy (X_train,y_train) na (X_train2,y_train2), (X_valid,y_valid).
    2. Dla ka?dego zestawu hiperparametr?w:
        1. Dla ka?dego podzia?u (X_train2,y_train2), (X_valid,y_valid):
            1. Tworzymy model = Model(zestaw_hiperparametr?w).
            2. Uczymy model na (X_train2,y_train2), testujemy na (X_valid,y_valid) i otrzymujemy score.
            3. (zestaw_hiperparametr?w, score) zapisujemy w tabelce.
    3. Dla ka?dego zestawu hiperparametr?w mamy trzy r??ne score z trzech podzia??w - u?redniamy je.
    4. Wybieramy ?rednio najlepszy zestaw hiperparametr?w.
    5. Tworzymy model = Model(?rednio_najlepszy_zestaw_hiperparametr?w) i uczymy na ca?ym (X_train,y_train).
    6. Testujemy model na (X_test,y_test) i otrzymujemy evaluation_score, zapisujemy go.
3. U?redniamy trzy evaluation_score, zwracamy ?redni? jako ostateczny score Modelu.

Uwaga - dla r??nych (X_train,y_train) mamy r??ne ostatecznie wybrane ?rednio_najlepszy_zestaw_hiperparametr?w, tak w?a?nie ma by? [prosz? si? zastanowi?, dlaczego].

Aby wytrenowa? metamodel:
1. Dzielimy trzy razy (X,y) na (X_train,y_train), (X_valid,y_valid).
2. Dla ka?dego zestawu hiperparametr?w:
    1. Dla ka?dego podzia?u (X_train,y_train), (X_valid,y_valid):
        1. Tworzymy model = Model(zestaw_hiperparametr?w).
        2. Uczymy model na (X_train,y_train), testujemy na (X_valid,y_valid) i otrzymujemy score.
        3. (zestaw_hiperparametr?w, score) zapisujemy w tabelce.
3. Dla ka?dego zestawu hiperparametr?w mamy trzy r??ne score z trzech podzia??w - u?redniamy je.
4. Wybieramy ?rednio najlepszy zestaw hiperparametr?w.
5. Tworzymy model = Model(?rednio_najlepszy_zestaw_hiperparametr?w) i uczymy na ca?ym (X,y).

Uwaga - teraz mamy tylko jeden ?rednio_najlepszy_zestaw_hiperparametr?w i jeden model.

W ten spos?b otrzymujemy:
1. model - ten model (wytrenowany metamodel) idzie do produkcji.
2. score - estymacja skuteczno?ci klasy Model jest do?? dobr? estymacj? tego, jak model poradzi sobie w przysz?o?ci.
3. ?rednio_najlepszy_zestaw_hiperparametr?w - informacyjnie mo?emy te? poda? znaleziony najlepszy zestaw hiperparametr?w, jednak?e w tym momencie nie r??ni si? on koncepcyjnie od wytrenowanego zestawu **parametr?w**; nie mo?emy twierdzi?, ?e te hiperparametry b?d? zawsze najlepsze, poniewa? ucz?c si? na innych danych mo?emy dosta? inne warto?ci ?rednio najlepszych hiperparametr?w.

In [ ]:
import matplotlib.pylab as plt
%matplotlib inline
import numpy as np
from numpy.random import RandomState
from itertools import product

# dane
from sklearn.datasets import fetch_mldata
# splity
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
# modele
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC, SVC
# metryki
from sklearn.metrics import accuracy_score

mnist = fetch_mldata('MNIST original')
X = mnist["data"]
y = mnist["target"]

# zawsze przed uczeniem/splitami prosz? zrobi? shuffle na danych!

Czego b?dziemy u?ywa?:
* http://scikit-learn.org/stable/modules/generated/sklearn.model_selection.StratifiedKFold.html
* http://scikit-learn.org/stable/modules/generated/sklearn.svm.LinearSVC.html
* http://scikit-learn.org/stable/modules/generated/sklearn.svm.SVC.html
* http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html
* http://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html
* http://yann.lecun.com/exdb/mnist/

Uwaga - modele b?d? si? uczy?y d?u?ej, ni? zwykle. Warto doda? w kodzie du?o print?w, ?eby na bie??co wiedzie?, co si? dzieje.

Na etapie pisania i testowania kodu mo?na te? zmniejszy? sobie losowo MNISTa np. do 5% przyk?ad?w, aby testy przebiega?y szybciej.

## ?wiczenie 1 [2 pkt]

Napisa? trzy generatory zestaw?w hiperparametr?w:
* generator grid search [robimy wsp?lnie za 0 pkt],
* generator k element?w random grid search,
* generator k element?w random search.

Hiperparametry podajemy jako s?ownik {nazwa_hiperparametru: lista_warto?ci/rozk?ad}.

Generator ma yieldowa? s?owniki {nazwa_hiperparametru: warto??_hiperparametru}

Rozk?ady prawdopodobie?stwa mo?ecie Pa?stwo podawa? w dowolny spos?b - np. jako pythonow? funkcj?, jako string z nazw? (a mo?liwe rozk?ady zakodowa? na sztywno w funkcji zwracaj?cej generatory) itp. Proponuj? zastosowa? konwencj? opisan? w nast?pnej kom?rce [czy kto? z Pa?stwa ma inny pomys?, jak podawa? rozk?ady?].

Losowe generatory maj? przyjmowa? random_state i dzia?a? deterministycznie przy ustalonym random_state (nale?y ustawi? w ?rodku seed generatora liczb losowych na random_state).


In [ ]:
# Tworzymy fabryki sampler?w, ka?da fabryka parametryzuje pewn? rodzin? rozk?ad?w prawdopodobie?stwa
# sampler pami?ta jeden konkretny rozk?ad prawdopodobie?stwa
# jedynym argumentem samplera jest obiekt numpy.random.RandomState, kt?ry oznaczamy rng
# sampler zwraca jedn? warto?? ze swojego rozk?adu
# poniewa? rng pami?ta sw?j stan, to mo?emy np. raz stworzy? go na pocz?tku i podawa? w p?tli

# rodzina rozk?ad?w jednostajnych na podanych listach element?w
# sampler losuje z rozk?adu jednostajnego na li?cie l
def uniform_from_list(l):
    def sampler(rng):
        return l[rng.randint(0,len(l))]
    return sampler

# rodzina rozk?ad?w jednostajnych na przedzia?ach liczb ca?kowitych
# sampler losuje z rozk?adu jednostajnego na podzbiorze liczb ca?kowitych od low (w??cznie) do high (wy??cznie)
def uniform_int_on_interval(low, high):
    def sampler(rng):
        return rng.randint(low,high)
    return sampler

# rodzina rozk?ad?w jednostajnych na przedzia?ach
# sampler losuje z rozk?adu jednostajnego na przedziale [low, high]
def uniform_on_interval(low, high):
    def sampler(rng):
        return rng.uniform(low, high)
    return sampler

# rodzina rozk?ad?w jednostajnych na przedzia?ach w wyk?adniku pot?gi liczby 10
# sampler losuje liczb? alpha z rozk?adu jednostajnego na przedziale [low, high], a nast?pnie zwraca 10^alpha
def log_uniform_on_interval(low, high):
    def sampler(rng):
        return 10.**rng.uniform(low, high)
    return sampler

# rodzina rozk?ad?w gaussa
# sampler losuje liczb? z rozk?adu N(mean, std^2)
def normal(mean, std):
    def sampler(rng):
        return rng.normal(loc=mean, scale=std)
    return sampler

# rodzina rozk?ad?w gaussa w wyk?adniku pot?gi liczby 10
# sampler losuje liczb? alpha z rozk?adu N(mean, std^2), a nast?pnie zwraca 10^alpha
def log_normal(mean, std):
    def sampler(rng):
        return 10**rng.normal(loc=mean, scale=std)
    return sampler

In [ ]:
print "Tworzymy obiekt klasy RandomState"
rng = RandomState(743)
s1 = normal(0.,1.)
s2 = log_uniform_on_interval(-4.,4.)
s3 = uniform_int_on_interval(-2,5)
print "Losujemy..."
for _ in xrange(4):
    print s1(rng), s2(rng), s3(rng)

print ""
print "Resetujemy RandomState"
rng = RandomState(743)
s1 = normal(0.,1.)
s2 = log_uniform_on_interval(-4.,4.)
s3 = uniform_int_on_interval(-2,5)
print "Losujemy..."
for _ in xrange(4):
    print s1(rng), s2(rng), s3(rng)

print ""
print "Resetujemy RandomState"
rng = RandomState(743)
s1 = normal(0.,1.)
s2 = log_uniform_on_interval(-4.,4.)
s3 = uniform_int_on_interval(-2,5)
print "Losujemy w innej kolejno?ci, wypisujemy w starej..."
for _ in xrange(4):
    _s3 = s3(rng)
    _s2 = s2(rng)
    _s1 = s1(rng)
    print _s1, _s2, _s3

print ""
print "samplery wsp??dziel? rng, kolejno?? losowania ma znaczenie!"

In [ ]:
def grid_search(grid):
    (keys, values_grid) = zip(*grid.iteritems())
    for values in product(*values_grid):
        yield dict(zip(keys, values))

def random_grid_search(grid, k=20, random_state=43):
    rng = RandomState(random_state) # ustalamy jeden wsp?lny rng
    # wysamplowa? k zestaw?w hiperparametr?w
    # ...
    yield dict_of_hyperparams

def random_search(grid, k=20, random_state=43):
    rng = RandomState(random_state) # ustalamy jeden wsp?lny rng
    (keys, samplers) = zip(*sorted(grid.iteritems())) # sortujemy klucze, kolejno?? samplowania jest wa?na!
    # wysamplowa? k zestaw?w hiperparametr?w
    # ...
    yield dict_of_hyperparams


In [ ]:
# assert dla grid_search
for d in grid_search({'C': [0.1, 1., 10., 100.], 'gamma': [0.0001, 0.0003, 0.001]}):
    print d

{'C': 0.1, 'gamma': 0.0001}<br/>
{'C': 0.1, 'gamma': 0.0003}<br/>
{'C': 0.1, 'gamma': 0.001}<br/>
{'C': 1.0, 'gamma': 0.0001}<br/>
{'C': 1.0, 'gamma': 0.0003}<br/>
{'C': 1.0, 'gamma': 0.001}<br/>
{'C': 10.0, 'gamma': 0.0001}<br/>
{'C': 10.0, 'gamma': 0.0003}<br/>
{'C': 10.0, 'gamma': 0.001}<br/>
{'C': 100.0, 'gamma': 0.0001}<br/>
{'C': 100.0, 'gamma': 0.0003}<br/>
{'C': 100.0, 'gamma': 0.001}

In [ ]:
# assert dla random_grid_search
# ma wyj?? tak samo, jak random_search, tylko z podanymi samplerami uniform_from_list
# prosz? sobie sprawdzi? we w?asnym zakresie, my uwierzymy na s?owo

In [ ]:
for d in random_search(
        {'C': log_uniform_on_interval(-5., 5.),
         'gamma': uniform_on_interval(0.,0.1),
         'nb_sth': uniform_from_list([1,4,9,13,22])},
        k=15,
        random_state=43):
    print d

{'C': 0.00014143134252521947, 'nb_sth': 9, 'gamma': 0.06090665392794814}<br/>
{'C': 0.0025462216096392796, 'nb_sth': 13, 'gamma': 0.03271390558111398}<br/>
{'C': 5.590022241698091e-05, 'nb_sth': 1, 'gamma': 0.004505286023886656}<br/>
{'C': 1.950465371399982e-05, 'nb_sth': 13, 'gamma': 0.0733748296280283}<br/>
{'C': 0.0002976304542098269, 'nb_sth': 13, 'gamma': 0.0011286104130902254}<br/>
{'C': 3.705521747936987e-05, 'nb_sth': 22, 'gamma': 0.08666486408992002}<br/>
{'C': 31.420766577465653, 'nb_sth': 22, 'gamma': 0.05808772319264447}<br/>
{'C': 5.842880444285076e-05, 'nb_sth': 1, 'gamma': 0.08432246942297046}<br/>
{'C': 16.058920736060898, 'nb_sth': 22, 'gamma': 0.04037700398666926}<br/>
{'C': 35065.59811614318, 'nb_sth': 22, 'gamma': 0.04457583608397189}<br/>
{'C': 6.683446823678905e-05, 'nb_sth': 13, 'gamma': 0.08970985799815344}<br/>
{'C': 965.8767858660282, 'nb_sth': 1, 'gamma': 0.0332329385020804}<br/>
{'C': 0.28068153065434537, 'nb_sth': 1, 'gamma': 0.08247676837174123}<br/>
{'C': 12207.52376193616, 'nb_sth': 13, 'gamma': 0.09101530757801567}<br/>
{'C': 0.00015294037493259258, 'nb_sth': 9, 'gamma': 0.09375550594407336}

## ?wiczenie 2 [0 pkt]

Zaimplementowa? algorytm model selection z u?yciem cross validation (StratifiedKFold w sklearn) zgodnie z opisem w sekcji "Podsumowanie".

In [ ]:
def skf_model_selection(Model, hyperparams_generator, X, y, metric, n_splits, random_state):
    """
    Model - klasa modelu
    hyperparams_generator - generator hiperparametr?w
    X,y - dane i etykiety
    metric - funkcja o sygnaturze metric(y_true, y_pred), kt?ra ocenia skuteczno?? nauczonego modelu
    n_splits - liczba split?w/fold?w w cross validation
    random_state - u?ywany wsz?dzie tam, gdzie trzeba
    """
    
    scores = []
    tested_hyperparams = []
    
    # Aby wytrenowa? metamodel:
    # 1. Dzielimy n_splits (X,y) na (X_train,y_train), (X_valid,y_valid).
        # dla przejrzysto?ci algorytmu zapami?tujemy indeksy podzia?u w li?cie indices
        # w normalnym kodzie trzeba np. wielokrotnie stworzy? generator skf g??biej w p?tli
    indices = []
    skf = StratifiedKFold(n_splits=n_splits, random_state=random_state, shuffle=True)
    for train_index, valid_index in skf.split(X,y):
        indices.append((train_index, valid_index))
    # 2. Dla ka?dego zestawu hiperparametr?w:
    for hyperparams in hyperparams_generator:
        # 1. Dla ka?dego podzia?u (X_train,y_train), (X_valid,y_valid):
        scores.append([])
        print "Testujemy hiperparametry:", hyperparams
        for i, (train_index, valid_index) in enumerate(indices):
            X_train, y_train = X[train_index], y[train_index]
            X_valid, y_valid = X[valid_index], y[valid_index]
            # 1. Tworzymy model = Model(zestaw_hiperparametr?w).
            model = Model(**hyperparams)
            # 2. Uczymy model na (X_train,y_train), testujemy na (X_valid,y_valid) i otrzymujemy score.
            print "Uczymy model - split " + str(i+1) + "/" + str(n_splits) + "..."
            model.fit(X_train, y_train)
            print "Liczymy score..."
            score = metric(y_valid, model.predict(X_valid)) # w zale?no?ci od metryki czasem u?ywa si? model.predict_proba, tu zak?adamy, ?e metryka jest typu 'accuracy'
            print "Train score:", metric(y_train, model.predict(X_train))
            print "Valid score:", score
            # 3. (zestaw_hiperparametr?w, score) zapisujemy w tabelce.
            scores[-1].append(score)
        tested_hyperparams.append(hyperparams)
    # 3. Dla ka?dego zestawu hiperparametr?w mamy trzy r??ne score z trzech podzia??w - u?redniamy je.
    avg_scores = [np.mean(scores_for_single_hyperparams) for scores_for_single_hyperparams in scores]
    # 4. Wybieramy ?rednio najlepszy zestaw hiperparametr?w.
    best_hyperparams = tested_hyperparams[np.argmax(avg_scores)]
    print "Najlepszy zestaw hiperparametr?w:", best_hyperparams
    # 5. Tworzymy model = Model(?rednio_najlepszy_zestaw_hiperparametr?w) i uczymy na ca?ym (X,y).
    best_model = Model(**best_hyperparams)
    print "Uczymy najlepszy model..."
    best_model.fit(X, y)
    # Uwaga - teraz mamy tylko jeden ?rednio_najlepszy_zestaw_hiperparametr?w i jeden model.
    return best_model, best_hyperparams


In [ ]:
# przetestujmy powy?sze na losowym podzbiorze MNISTa

idx = np.random.RandomState(743).permutation(len(y))
_X = X[idx[:3000]]
_y = y[idx[:3000]]

best_model, best_hyperparams = skf_model_selection(
    Model=KNeighborsClassifier,
    hyperparams_generator=grid_search({'n_neighbors': [2,5,8], 'leaf_size': [20,30,40]}),
    X=_X, y=_y,
    metric=accuracy_score,
    n_splits=2,
    random_state=43)

## ?wiczenie 3 [4 pkt]

Zaimplementowa? algorytm model selection z u?yciem podw?jnej cross validation (StratifiedKFold w sklearn) zgodnie z opisem w sekcji "Podsumowanie".

In [ ]:
def double_skf_model_evaluation(Model, generator_function, generator_function_kwargs, X, y, metric, selection_n_splits, evaluation_n_splits, random_state):
    """
    Model - klasa modelu
    generator_function, generator_function_kwargs - chcemy zrobi? co? w stylu:
        for hyperparams in generator_function(**generator_function_kwargs):
            ...
        nie podajemy wprost generatora, bo trzeba go u?y? wielokrotnie, a generator?w (chyba?) nie da si? kopiowa?
    X,y - dane i etykiety
    metric - funkcja o sygnaturze metric(y_true, y_pred), kt?ra ocenia skuteczno?? nauczonego modelu
    selection/evaluation_n_splits - liczba split?w/fold?w w odpowiednim cross validation
    random_state - u?ywany wsz?dzie tam, gdzie trzeba
    """
    # 1. Dzielimy evaluation_n_splits razy (X,y) na (X_train,y_train), (X_test,y_test).
    # 2. Dla ka?dego takiego podzia?u:
        # 1. Dzielimy selection_n_splits razy (X_train,y_train) na (X_train2,y_train2), (X_valid,y_valid).
        # 2. Dla ka?dego zestawu hiperparametr?w:
            # 1. Dla ka?dego podzia?u (X_train2,y_train2), (X_valid,y_valid):
                # 1. Tworzymy model = Model(zestaw_hiperparametr?w).
                # 2. Uczymy model na (X_train2,y_train2), testujemy na (X_valid,y_valid) i otrzymujemy score.
                # 3. (zestaw_hiperparametr?w, score) zapisujemy w tabelce.
        # 3. Dla ka?dego zestawu hiperparametr?w mamy trzy r??ne score z trzech podzia??w - u?redniamy je.
        # 4. Wybieramy ?rednio najlepszy zestaw hiperparametr?w.
        # 5. Tworzymy model = Model(?rednio_najlepszy_zestaw_hiperparametr?w) i uczymy na ca?ym (X_train,y_train).
        # 6. Testujemy model na (X_test,y_test) i otrzymujemy evaluation_score, zapisujemy go.
    # 3. U?redniamy trzy evaluation_score, zwracamy ?redni? jako ostateczny score Modelu.
    return score

## ?wiczenie 4 [2 pkt]

Przy u?yciu powy?szych funkcji wytrenowa? porz?dnie na danych MNIST jeden (dowolnie wybrany) z poni?szych model?w:
* LinearSVM,
* SVM,
* RandomForest,
* KNN.

Zwr?ci? model i jego estymowany score.

[Mo?emy wsp?lnie zastanowi? si? nad sensownym doborem grid?w/rozk?ad?w na hiperparametrach.]

## Data augmentation

Model zawsze nauczy si? lepiej na wi?kszym zbiorze danych. Przez data augmentation rozumiemy ca?y zbi?r technik, kt?re pozwalaj? "sztucznie" niskim kosztem powi?kszy? dataset. Oczywi?cie musimy zna? etykiety nowowygenerowanych danych. Zazwyczaj dzia?amy w nast?puj?cy spos?b: definiujemy operacj?, kt?r? mo?emy zadzia?a? na pojedynczy $\mathbf{x}$ bez zmiany odpowiadaj?cego mu $y$, a nast?pnie generujemy z jednego $\mathbf{x}$ du?o wersji i ka?dej przypisujemy to samo $y$.

Kilka prostych przyk?ad?w:
* rozpoznawanie mowy ($\mathbf{x}$ - zapis d?wi?kowy, $y$ - tekst wypowiadany przez lektora) - zmieniamy szybko?? m?wienia, zmieniamy wysoko?? d?wi?ku itp.
* klasyfikacja obrazk?w - obracamy obrazki, odbijamy symetrycznie, dodajemy szum, zmieniamy nieco palet? barw itp.

Spr?bujmy zobaczy? dla przyk?adu, co si? stanie, gdy dodamy gaussowski szum do MNISTa.

In [ ]:
mnist = fetch_mldata('MNIST original')
X = mnist["data"]
y = mnist["target"]
idx = np.random.permutation(len(y))
X = X[idx[:3000]]
y = y[idx[:3000]]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=43)

model = KNeighborsClassifier(n_neighbors=1)
print "fitting KNN (no augmentation)"
model.fit(X_train, y_train)
print "train accuracy:", accuracy_score(y_train, model.predict(X_train))
print "test accuracy:", accuracy_score(y_test, model.predict(X_test))

In [ ]:
print "Augmenting data..."
_X = [X.copy()]
_y = [y.copy()]
for i in xrange(7):
    _X.append(X.copy() + np.random.normal(scale=3, size=X.shape))
    _y.append(y.copy())
X_aug = np.vstack(_X)
y_aug = np.hstack(_y)

plt.imshow(X[-1].reshape(28,28), cmap="gray")
plt.show()
plt.imshow(X_aug[-1].reshape(28,28), cmap="gray")
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_aug, y_aug, test_size=0.33, random_state=43)

model = KNeighborsClassifier(n_neighbors=1)
print "fitting KNN (with augmentation)"
model.fit(X_train, y_train)
print "train accuracy:", accuracy_score(y_train, model.predict(X_train))
print "test accuracy:", accuracy_score(y_test, model.predict(X_test))

Hurra! Po dodaniu stosunkowo prostej augmentacji danych nasz model nauczy? si? perfekcyjnie generalizowa? na zbi?r testowy! I nie musieli?my tu stosowa? ?adnej skomplikowanej sieci neuronowej, wystarczy? KNN.

## ?wiczenie 5 (4 pkt)

Wyt?umaczy?, na czym polega b??d w powy?szym rozumowaniu.